In [ ]:
# Bangla ELECTRA Fine-tuning for Multi-label Cyberbullying Detection

# Install Required Packages and Setup Google Drive

# !pip install -q transformers datasets torch scikit-learn pandas numpy matplotlib seaborn

# Mount Google Drive for automatic saving
from google.colab import drive
drive.mount('/content/drive')

# print("All packages installed successfully!")
print("Google Drive mounted successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 97.0 MB/s eta 0:00:00
Mounted at /content/drive
All packages installed successfully!
Google Drive mounted successfully!


In [ ]:
# Import all the libraries and setup project configuration
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import (
    XLMRobertaTokenizer,
    XLMRobertaModel,
    get_linear_schedule_with_warmup
)
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, multilabel_confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import os
import json
import shutil
from datetime import datetime

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# Project Configuration
PROJECT_CONFIG = {
    'model_name': 'xlm-roberta',
    'model_path': 'xlm-roberta-base',
    'project_name': 'XLM-RoBERTa-Cyberbullying',
    'experiment_date': datetime.now().strftime("%Y%m%d_%H%M%S"),
    'num_folds': 5,
    'momentum': 0.9,
    'dropout': 0.3,
    'max_length': 128
}

# Google Drive Setup
DRIVE_BASE_PATH = '/content/drive/MyDrive'
PROJECT_DRIVE_PATH = f"{DRIVE_BASE_PATH}/{PROJECT_CONFIG['project_name']}"

# Create project directory structure in Google Drive
def setup_drive_directories():
    """Create necessary directories in Google Drive for the project"""
    directories = [
        PROJECT_DRIVE_PATH,
        f"{PROJECT_DRIVE_PATH}/experiments",
    ]

    for directory in directories:
        os.makedirs(directory, exist_ok=True)
        print(f"Created/verified directory: {directory}")

setup_drive_directories()

# Session directory for current experiment
SESSION_ID = f"{PROJECT_CONFIG['model_name']}_{PROJECT_CONFIG['experiment_date']}"
SESSION_DRIVE_PATH = f"{PROJECT_DRIVE_PATH}/experiments/{SESSION_ID}"
os.makedirs(SESSION_DRIVE_PATH, exist_ok=True)

print(f"Session directory: {SESSION_DRIVE_PATH}")

# Save project configuration
config_file = f"{SESSION_DRIVE_PATH}/project_config.json"
with open(config_file, 'w') as f:
    json.dump(PROJECT_CONFIG, f, indent=2)

print(f"Project configuration saved to: {config_file}")

Created/verified directory: /content/drive/MyDrive/XLM-RoBERTa-Cyberbullying
Created/verified directory: /content/drive/MyDrive/XLM-RoBERTa-Cyberbullying/experiments
Session directory: /content/drive/MyDrive/XLM-RoBERTa-Cyberbullying/experiments/xlm-roberta_20250823_210405
Project configuration saved to: /content/drive/MyDrive/XLM-RoBERTa-Cyberbullying/experiments/xlm-roberta_20250823_210405/project_config.json


In [ ]:
# Load dataset and automatically backup to Google Drive

from google.colab import files

print("Please upload your dataset file:")
uploaded = files.upload()

# Get the filename of the uploaded file
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print("Dataset loaded successfully!")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# Backup original dataset to Google Drive
dataset_backup_path = f"{SESSION_DRIVE_PATH}/original_dataset.csv"
df.to_csv(dataset_backup_path, index=False)
print(f"Dataset backed up to: {dataset_backup_path}")

# Display first few rows
print("\nFirst 5 rows of the dataset:")
print(df.head())

# Generate and save dataset report
dataset_report = {
    'shape': df.shape,
    'columns': df.columns.tolist(),
    'missing_values': df.isnull().sum().to_dict(),
    'data_types': df.dtypes.to_dict(),
    'memory_usage': df.memory_usage(deep=True).sum()
}

report_file = f"{SESSION_DRIVE_PATH}/dataset_report.json"
with open(report_file, 'w') as f:
    json.dump(dataset_report, f, indent=2, default=str)

print(f"Dataset report saved to: {report_file}")

Please upload your dataset file:


Saving 1_Multilablel_Cyberbully_Data.csv to 1_Multilablel_Cyberbully_Data.csv
Dataset loaded successfully!
Dataset shape: (12546, 8)
Columns: ['Gender', 'Profession', 'comment', 'bully', 'sexual', 'religious', 'threat', 'spam']
Dataset backed up to: /content/drive/MyDrive/XLM-RoBERTa-Cyberbullying/experiments/xlm-roberta_20250823_210405/original_dataset.csv

First 5 rows of the dataset:
   Gender Profession                                            comment  \
0  female     dancer                              এই দেশে এইসব কি হচ্ছে   
1  female     dancer                                        মানে কি বলব   
2  female     dancer                                ভাই ভিডিও ফুল প্লিজ   
3  female     dancer  নিজের খরচ নিজেই চালাতে পারবেন এমন ভালো একটা জব...   
4  female     dancer                  ভিডিও কলে রেগুলার কাজ করতে পারবেন   

   bully  sexual  religious  threat  spam  
0      0       0          0       0     0  
1      0       0          0       0     0  
2      0       1          0

In [ ]:
# Clean and prepare the dataset for training

print("Preprocessing the dataset...")

# Drop gender and profession columns as mentioned
df_clean = df.drop(['Gender', 'Profession'], axis=1)
print(f"Dropped Gender and Profession columns. New shape: {df_clean.shape}")

# Check for missing values
missing_values_report = df_clean.isnull().sum()
print(f"\nMissing values check:")
print(missing_values_report)

# Display label distribution
label_columns = ['bully', 'sexual', 'religious', 'threat', 'spam']
label_distribution = {}

print(f"\nLabel distribution:")
for label in label_columns:
    positive_count = df_clean[label].sum()
    percentage = positive_count/len(df_clean)*100
    label_distribution[label] = {
        'positive_samples': int(positive_count),
        'percentage': float(percentage)
    }
    print(f"{label}: {positive_count} positive samples ({percentage:.2f}%)")

# Save preprocessing report
preprocessing_report = {
    'original_shape': df.shape,
    'cleaned_shape': df_clean.shape,
    'dropped_columns': ['Gender', 'Profession'],
    'label_columns': label_columns,
    'label_distribution': label_distribution,
    'missing_values': missing_values_report.to_dict()
}

preprocessing_file = f"{SESSION_DRIVE_PATH}/preprocessing_report.json"
with open(preprocessing_file, 'w') as f:
    json.dump(preprocessing_report, f, indent=2, default=str)

# Save cleaned dataset
cleaned_dataset_path = f"{SESSION_DRIVE_PATH}/cleaned_dataset.csv"
df_clean.to_csv(cleaned_dataset_path, index=False)

print(f"Preprocessing report saved to: {preprocessing_file}")
print(f"Cleaned dataset saved to: {cleaned_dataset_path}")

# Show some sample comments and their labels
print(f"\nSample comments with labels:")
for i in range(3):
    comment = df_clean.iloc[i]['comment']
    labels = [df_clean.iloc[i][col] for col in label_columns]
    print(f"Comment: {comment}")
    print(f"Labels: {dict(zip(label_columns, labels))}")
    print("-" * 50)

Preprocessing the dataset...
Dropped Gender and Profession columns. New shape: (12546, 6)

Missing values check:
comment      0
bully        0
sexual       0
religious    0
threat       0
spam         0
dtype: int64

Label distribution:
bully: 8052 positive samples (64.18%)
sexual: 2059 positive samples (16.41%)
religious: 1607 positive samples (12.81%)
threat: 1422 positive samples (11.33%)
spam: 1089 positive samples (8.68%)
Preprocessing report saved to: /content/drive/MyDrive/XLM-RoBERTa-Cyberbullying/experiments/xlm-roberta_20250823_210405/preprocessing_report.json
Cleaned dataset saved to: /content/drive/MyDrive/XLM-RoBERTa-Cyberbullying/experiments/xlm-roberta_20250823_210405/cleaned_dataset.csv

Sample comments with labels:
Comment: এই দেশে এইসব কি হচ্ছে
Labels: {'bully': np.int64(0), 'sexual': np.int64(0), 'religious': np.int64(0), 'threat': np.int64(0), 'spam': np.int64(0)}
--------------------------------------------------
Comment: মানে কি বলব
Labels: {'bully': np.int64(0), 

In [ ]:
# Create a custom dataset class for our multi-label classification task

class CyberbullyingDataset(Dataset):
    """
    Custom Dataset class for loading cyberbullying data

    1. Load comments and labels in batches
    2. Tokenize text using XLM Roberta tokenizer
    3. Convert labels to tensors
    """

    def __init__(self, comments, labels, tokenizer, max_length=128):
        """
        Initialize the dataset

        Args:
            comments: List of text comments
            labels: List of label arrays (multi-label)
            tokenizer: XLM Roberta tokenizer
            max_length: Maximum sequence length for tokenization
        """
        self.comments = comments
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        """Return the total number of samples"""
        return len(self.comments)

    def __getitem__(self, idx):
        """
        Get a single sample from the dataset

        This method:
        1. Takes a comment and tokenizes it
        2. Converts labels to tensor format
        3. Returns the processed data
        """
        comment = str(self.comments[idx])
        labels = self.labels[idx]

        # Tokenize the comment
        encoding = self.tokenizer(
            comment,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(labels, dtype=torch.float)
        }

print("Custom Dataset class created successfully!")

Custom Dataset class created successfully!


In [ ]:
from transformers import XLMRobertaModel

class XLMRobertaMultiLabelClassifier(nn.Module):
    """
    Multi-label classifier based on XLM Roberta

    This model:
    1. Uses pre-trained XLM Roberta as base
    2. Adds a custom classification head for 5 labels
    3. Uses pooling to get sentence-level representation
    """
    def __init__(self, model_name, num_labels):
      super(XLMRobertaMultiLabelClassifier, self).__init__()

      # Load pre-trained XLM-RoBERTa model
      self.roberta = XLMRobertaModel.from_pretrained(model_name)

      # Custom classification head for multi-label classification
      self.classifier = nn.Sequential(
          nn.Linear(self.roberta.config.hidden_size, 256),
          nn.ReLU(),
          nn.Dropout(0.1),
          nn.Linear(256, num_labels)
        )

    def forward(self, input_ids, attention_mask):
        """
        Forward pass through the model

        Args:
            input_ids: Tokenized input sequences
            attention_mask: Attention masks for input

        Returns:
            Logits for each label (batch_size, num_labels)
        """
        # Get RoBERTa outputs
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)

        # Get the [CLS] token representation (first token)
        # This represents the entire sequence
        cls_output = outputs.last_hidden_state[:, 0, :]  # Shape: (batch_size, hidden_size)

        # Pass through classification head
        logits = self.classifier(cls_output)  # Shape: (batch_size, num_labels)

        return logits

print("Multi-label XLM Roberta classifier created!")

Multi-label XLM Roberta classifier created!


In [ ]:
# Training functions with automatic saving capabilities
def calculate_class_weights(labels):
    """Calculate class weights for imbalanced dataset"""
    pos_counts = np.sum(labels, axis=0)
    neg_counts = len(labels) - pos_counts
    weights = neg_counts / pos_counts
    return torch.FloatTensor(weights)

def calculate_metrics(y_true, y_pred):
    """Calculate evaluation metrics for multi-label classification"""
    # Convert probabilities to binary predictions (threshold = 0.5)
    y_pred_binary = (y_pred > 0.5).astype(int)

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred_binary)
    precision = precision_score(y_true, y_pred_binary, average='macro', zero_division=0)
    recall = recall_score(y_true, y_pred_binary, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred_binary, average='macro', zero_division=0)

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

# def train_epoch(model, dataloader, optimizer, scheduler, device):
def train_epoch(model, dataloader, optimizer, scheduler, device, class_weights=None):
    """Train the model for one epoch"""
    model.train()
    total_loss = 0
    progress_bar = tqdm(dataloader, desc='Training')

    for batch in progress_bar:
        # Move batch to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Clear gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(input_ids, attention_mask)

        # Calculate loss - use weighted if class_weights provided
        if class_weights is not None:
            loss_fn = nn.BCEWithLogitsLoss(pos_weight=class_weights)
        else:
            loss_fn = nn.BCEWithLogitsLoss()
        loss = loss_fn(outputs, labels)

        # Backward pass
        loss.backward()

        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        progress_bar.set_postfix({'loss': loss.item()})

    return total_loss / len(dataloader)

def evaluate_model(model, dataloader, device):
    """Evaluate the model on validation/test data"""
    model.eval()
    total_loss = 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        progress_bar = tqdm(dataloader, desc='Evaluating')

        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Forward pass
            outputs = model(input_ids, attention_mask)

            # Calculate loss (use simple BCE for evaluation)
            loss_fn = nn.BCEWithLogitsLoss()
            loss = loss_fn(outputs, labels)
            total_loss += loss.item()

            # Apply sigmoid to get probabilities
            predictions = torch.sigmoid(outputs)

            # Store predictions and labels
            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculate metrics
    avg_loss = total_loss / len(dataloader)
    metrics = calculate_metrics(np.array(all_labels), np.array(all_predictions))
    metrics['loss'] = avg_loss

    return metrics

def save_session_data(results_df, session_info, force_save=False):
    """
    Save current session data to Google Drive

    Args:
        results_df: DataFrame with experiment results
        session_info: Dictionary with session information
        force_save: Force save even if no new results
    """
    try:
        # Save results
        results_file = f"{SESSION_DRIVE_PATH}/results.csv"
        results_df.to_csv(results_file, index=False)

        # Save session info
        session_file = f"{SESSION_DRIVE_PATH}/session_info.json"
        with open(session_file, 'w') as f:
            json.dump(session_info, f, indent=2, default=str)

        # Create backup with timestamp
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        backup_results = f"{SESSION_DRIVE_PATH}/backup_results_{timestamp}.csv"
        results_df.to_csv(backup_results, index=False)

        print(f"Session data saved to Google Drive:")
        print(f"   - Results: {results_file}")
        print(f"   - Session Info: {session_file}")
        print(f"   - Backup: {backup_results}")

        return True
    except Exception as e:
        print(f"Error saving session data: {str(e)}")
        return False

print("Enhanced training functions with auto-save created!")

Enhanced training functions with auto-save created!


In [ ]:
# Define the hyperparameter combinations to test

# Hyperparameter combinations from your table
experiment_configs = [
    # Batch size 16 experiments
    {'batch_size': 16, 'learning_rate': 3e-5, 'num_epochs': 10},
    # {'batch_size': 16, 'learning_rate': 2e-5, 'num_epochs': 20},
    # {'batch_size': 16, 'learning_rate': 2e-5, 'num_epochs': 30},

    # Batch size 32 experiments
    {'batch_size': 32, 'learning_rate': 3e-5, 'num_epochs': 10},
    # {'batch_size': 32, 'learning_rate': 2e-5, 'num_epochs': 20},
    # {'batch_size': 32, 'learning_rate': 2e-5, 'num_epochs': 30},

    # Batch size 64 experiments
    {'batch_size': 64, 'learning_rate': 3e-5, 'num_epochs': 10},
    # {'batch_size': 64, 'learning_rate': 2e-5, 'num_epochs': 20},
    # {'batch_size': 64, 'learning_rate': 2e-5, 'num_epochs': 30},
]

print(f"Set up {len(experiment_configs)} experiment configurations")

# Save experiment configurations to Drive
configs_file = f"{SESSION_DRIVE_PATH}/experiment_configurations.json"
with open(configs_file, 'w') as f:
    json.dump(experiment_configs, f, indent=2)
print(f"Experiment configurations saved to: {configs_file}")

Set up 3 experiment configurations
Experiment configurations saved to: /content/drive/MyDrive/XLM-RoBERTa-Cyberbullying/experiments/xlm-roberta_20250823_210405/experiment_configurations.json


In [ ]:
# =============================================================================
# CELL 9: Data Preparation with Drive Save (Enhanced)
# =============================================================================
# Prepare the data for cross-validation experiments

print("Preparing data for experiments...")

# Extract features and labels
comments = df_clean['comment'].values
labels = df_clean[label_columns].values

print(f"Prepared {len(comments)} comments with {len(label_columns)} labels each")

# Initialize tokenizer
MODEL_NAME = PROJECT_CONFIG['model_path']
tokenizer = XLMRobertaTokenizer.from_pretrained(MODEL_NAME)

print(f"Loaded tokenizer for {MODEL_NAME}")

# Create results storage
results_df = pd.DataFrame(columns=[
    'Batch_size', 'Learning_rate', 'Num_epochs', 'Num_folds', 'Momentum',
    'Dropout', 'Accuracy', 'Precision', 'Recall', 'F1_Score'
])

# Initialize session info
session_info = {
    'session_id': SESSION_ID,
    'model_name': PROJECT_CONFIG['model_name'],
    'model_path': PROJECT_CONFIG['model_path'],
    'start_time': datetime.now().isoformat(),
    'total_experiments': len(experiment_configs),
    'completed_experiments': 0,
    'current_status': 'initialized',
    'dataset_info': {
        'num_samples': len(comments),
        'num_labels': len(label_columns),
        'label_columns': label_columns
    }
}

# Save initial session data
save_session_data(results_df, session_info, force_save=True)

print("Ready to start experiments!")

Preparing data for experiments...
Prepared 12546 comments with 5 labels each


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

Loaded tokenizer for xlm-roberta-base
Session data saved to Google Drive:
   - Results: /content/drive/MyDrive/XLM-RoBERTa-Cyberbullying/experiments/xlm-roberta_20250823_210405/results.csv
   - Session Info: /content/drive/MyDrive/XLM-RoBERTa-Cyberbullying/experiments/xlm-roberta_20250823_210405/session_info.json
   - Backup: /content/drive/MyDrive/XLM-RoBERTa-Cyberbullying/experiments/xlm-roberta_20250823_210405/backup_results_20250823_210446.csv
Ready to start experiments!


In [ ]:
# Run experiments with comprehensive saving and resume capability

def create_experiment_id(config):
    """Create unique ID for each experiment configuration"""
    # Handle scientific notation properly
    lr_str = f"{config['learning_rate']:.0e}".replace('-0', '-').replace('+0', '+')
    return f"B{config['batch_size']}_LR{lr_str}_E{config['num_epochs']}"


def run_single_experiment(config, comments, labels, tokenizer, device, exp_number, total_experiments):
    """Run a single experiment configuration with k-fold cross-validation"""
    exp_id = create_experiment_id(config)
    print(f"\nRunning experiment {exp_number}/{total_experiments}: {exp_id}")
    print(f"   Batch Size={config['batch_size']}, LR={config['learning_rate']}, Epochs={config['num_epochs']}")

    # Create experiment-specific directory
    exp_dir = f"{SESSION_DRIVE_PATH}/individual_experiments/{exp_id}"
    os.makedirs(exp_dir, exist_ok=True)

    # Initialize k-fold cross-validation
    kfold = KFold(n_splits=PROJECT_CONFIG['num_folds'], shuffle=True, random_state=42)

    fold_results = []
    experiment_log = {
        'experiment_id': exp_id,
        'config': config,
        'start_time': datetime.now().isoformat(),
        'folds': []
    }

    for fold, (train_idx, val_idx) in enumerate(kfold.split(comments)):
        fold_start_time = datetime.now()
        print(f"Fold {fold + 1}/{PROJECT_CONFIG['num_folds']}")

        # Split data
        train_comments = comments[train_idx]
        train_labels = labels[train_idx]
        val_comments = comments[val_idx]
        val_labels = labels[val_idx]

        # Calculate class weights for this fold
        class_weights = calculate_class_weights(train_labels).to(device)
        print(f"Class weights for fold {fold+1}: {class_weights}")

        # Create datasets
        train_dataset = CyberbullyingDataset(train_comments, train_labels, tokenizer, PROJECT_CONFIG['max_length'])
        val_dataset = CyberbullyingDataset(val_comments, val_labels, tokenizer, PROJECT_CONFIG['max_length'])

        # Create data loaders
        train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False)

        # Initialize model
        model = XLMRobertaMultiLabelClassifier(MODEL_NAME, len(label_columns))
        model.to(device)

        print(f"Model: {MODEL_NAME}")
        print(f"Number of parameters: {sum(p.numel() for p in model.parameters())}")
        print(f"Tokenizer vocab size: {tokenizer.vocab_size}")

        # Initialize optimizer and scheduler
        optimizer = AdamW(
            model.parameters(),
            lr=config['learning_rate'],
            weight_decay=0.01,  # Add this line
            eps=1e-8  # Add this line
            )

        total_steps = len(train_loader) * config['num_epochs']
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(0.1 * total_steps), # Warmup Steps 10% of total steps
            num_training_steps=total_steps
        )

        # Training loop
        best_f1 = 0
        patience = 5  # Early stopping patience
        patience_counter = 0
        epoch_history = []

        for epoch in range(config['num_epochs']):
            train_loss = train_epoch(model, train_loader, optimizer, scheduler, device, class_weights)
            val_metrics = evaluate_model(model, val_loader, device)

            epoch_data = {
                'epoch': epoch + 1,
                'train_loss': train_loss,
                'val_metrics': val_metrics
            }
            epoch_history.append(epoch_data)

            print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val F1={val_metrics['f1']:.4f}")

            if val_metrics['f1'] > best_f1:
                best_f1 = val_metrics['f1']
                best_metrics = val_metrics.copy()
                patience_counter = 0

                # Save best model for this fold
                torch.save({
                    'model_state_dict': model.state_dict(),
                    'config': config,
                    'fold': fold,
                    'epoch': epoch + 1,
                    'metrics': best_metrics
                }, f"{exp_dir}/best_model_fold_{fold}.pth")

            else:
              patience_counter += 1

            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

        fold_duration = (datetime.now() - fold_start_time).total_seconds()

        fold_info = {
            'fold': fold,
            'duration_seconds': fold_duration,
            'best_metrics': best_metrics,
            'epoch_history': epoch_history,
            'stopped_early': patience_counter >= patience
        }
        experiment_log['folds'].append(fold_info)

        fold_results.append(best_metrics)
        print(f"Fold {fold + 1} completed. Best F1: {best_f1:.4f} (Duration: {fold_duration:.1f}s)")

    # Calculate average metrics across folds
    avg_metrics = {}
    for key in fold_results[0].keys():
        if key != 'loss':  # Don't average loss
            avg_metrics[key] = np.mean([result[key] for result in fold_results])

    # Complete experiment log
    experiment_log['end_time'] = datetime.now().isoformat()
    experiment_log['duration_seconds'] = (datetime.now() - datetime.fromisoformat(experiment_log['start_time'])).total_seconds()
    experiment_log['avg_metrics'] = avg_metrics
    experiment_log['std_metrics'] = {key: np.std([result[key] for result in fold_results]) for key in avg_metrics.keys()}

    # Save experiment log
    log_file = f"{exp_dir}/experiment_log.json"
    with open(log_file, 'w') as f:
        json.dump(experiment_log, f, indent=2, default=str)

    return avg_metrics

# Load existing results if available
RESULTS_FILE = f"{SESSION_DRIVE_PATH}/results.csv"
if os.path.exists(RESULTS_FILE):
    print(f"Loading existing results from {RESULTS_FILE}")
    results_df = pd.read_csv(RESULTS_FILE)
    completed_experiments = set()
    for _, row in results_df.iterrows():
        # Handle scientific notation in learning rate
        lr_str = f"{row['Learning_rate']:.0e}".replace('-0', '-').replace('+0', '+')
        exp_id = f"B{int(row['Batch_size'])}_LR{lr_str}_E{int(row['Num_epochs'])}"
        completed_experiments.add(exp_id)
    print(f"Found {len(completed_experiments)} completed experiments")
else:
    print("No existing results found. Starting fresh.")
    results_df = pd.DataFrame(columns=[
        'Batch_size', 'Learning_rate', 'Num_epochs', 'Num_folds', 'Momentum',
        'Dropout', 'Accuracy', 'Precision', 'Recall', 'F1_Score'
    ])
    completed_experiments = set()

# Filter out completed experiments
remaining_configs = []
for config in experiment_configs:
    exp_id = create_experiment_id(config)
    if exp_id not in completed_experiments:
        remaining_configs.append(config)

print(f"Total experiments: {len(experiment_configs)}")
print(f"Completed experiments: {len(completed_experiments)}")
print(f"Remaining experiments: {len(remaining_configs)}")

# Set how many experiments to run in this session
MAX_EXPERIMENTS_PER_SESSION = 4  # Adjust this based on your time limit
experiments_to_run = remaining_configs[:MAX_EXPERIMENTS_PER_SESSION]

print(f"Running {len(experiments_to_run)} experiments in this session:")
for i, config in enumerate(experiments_to_run):
    exp_id = create_experiment_id(config)
    print(f"  {i+1}. {exp_id}")

# Update session info
session_info.update({
    'current_status': 'running',
    'experiments_this_session': len(experiments_to_run),
    'session_start_time': datetime.now().isoformat()
})

# Run experiments with enhanced saving
session_results = []
start_time = datetime.now()

for i, config in enumerate(experiments_to_run):
    try:
        exp_id = create_experiment_id(config)
        print(f"\n{'='*60}")
        print(f"Session Progress: {i+1}/{len(experiments_to_run)}")
        print(f"Elapsed time: {datetime.now() - start_time}")
        print(f"Experiment ID: {exp_id}")

        # Run experiment
        avg_metrics = run_single_experiment(
            config, comments, labels, tokenizer, device,
            len(results_df) + i + 1, len(experiment_configs)
        )

        # Store results
        result_row = {
            'Batch_size': config['batch_size'],
            'Learning_rate': config['learning_rate'],
            'Num_epochs': config['num_epochs'],
            'Num_folds': PROJECT_CONFIG['num_folds'],
            'Momentum': PROJECT_CONFIG['momentum'],
            'Dropout': PROJECT_CONFIG['dropout'],
            'Accuracy': avg_metrics['accuracy'],
            'Precision': avg_metrics['precision'],
            'Recall': avg_metrics['recall'],
            'F1_Score': avg_metrics['f1']
        }

        # Add to results dataframe
        results_df = pd.concat([results_df, pd.DataFrame([result_row])], ignore_index=True)
        session_results.append(result_row)

        # Update session info
        session_info['completed_experiments'] = len(results_df)
        session_info['last_completed_experiment'] = exp_id
        session_info['last_update'] = datetime.now().isoformat()

        # Save results immediately after each experiment
        save_session_data(results_df, session_info)

        print(f"Experiment {exp_id} completed successfully!")
        print(f"   Results: Acc={avg_metrics['accuracy']:.4f}, "
              f"Prec={avg_metrics['precision']:.4f}, "
              f"Rec={avg_metrics['recall']:.4f}, "
              f"F1={avg_metrics['f1']:.4f}")
        print(f"Results automatically saved to Google Drive")

    except Exception as e:
        print(f"Experiment {exp_id} failed: {str(e)}")
        # Save error info
        error_dir = f"{SESSION_DRIVE_PATH}/errors"
        os.makedirs(error_dir, exist_ok=True)

        error_info = {
            'experiment_id': exp_id,
            'error': str(e),
            'time': datetime.now().isoformat(),
            'config': config
        }

        with open(f'{error_dir}/error_{exp_id}.json', 'w') as f:
            json.dump(error_info, f, indent=2)
        continue

# Final session summary
end_time = datetime.now()
session_duration = end_time - start_time

session_info.update({
    'current_status': 'completed' if len(remaining_configs) == len(experiments_to_run) else 'paused',
    'session_end_time': end_time.isoformat(),
    'session_duration_seconds': session_duration.total_seconds(),
    'session_duration_human': str(session_duration)
})

# Final save
save_session_data(results_df, session_info, force_save=True)

print(f"\n{'='*60}")
print(f"SESSION SUMMARY")
print(f"{'='*60}")
print(f"Session duration: {session_duration}")
print(f"Experiments completed this session: {len(session_results)}")
print(f"Total experiments completed: {len(results_df)}/{len(experiment_configs)}")
print(f"Remaining experiments: {len(experiment_configs) - len(results_df)}")

if len(results_df) < len(experiment_configs):
    print(f"\nTO CONTINUE LATER:")
    print(f"1. Re-run this cell to continue from where you left off")
    print(f"2. Adjust MAX_EXPERIMENTS_PER_SESSION if needed")
    print(f"3. All results are automatically saved to Google Drive")
else:
    print(f"\nALL EXPERIMENTS COMPLETED!")
    print(f"Proceed to analysis cells")

print(f"\nGoogle Drive Files:")
print(f"  - Main results: {RESULTS_FILE}")
print(f"  - Session info: {SESSION_DRIVE_PATH}/session_info.json")
print(f"  - Individual experiments: {SESSION_DRIVE_PATH}/individual_experiments/")
print(f"  - Project directory: {PROJECT_DRIVE_PATH}")

# Display current results preview
if len(results_df) > 0:
    print(f"\nCurrent Results Preview (showing last 5):")
    print(results_df.tail().round(4))

Loading existing results from /content/drive/MyDrive/XLM-RoBERTa-Cyberbullying/experiments/xlm-roberta_20250823_210405/results.csv
Found 0 completed experiments
Total experiments: 3
Completed experiments: 0
Remaining experiments: 3
Running 3 experiments in this session:
  1. B16_LR3e-5_E10
  2. B32_LR3e-5_E10
  3. B64_LR3e-5_E10

Session Progress: 1/3
Elapsed time: 0:00:00.000637
Experiment ID: B16_LR3e-5_E10

Running experiment 1/3: B16_LR3e-5_E10
   Batch Size=16, LR=3e-05, Epochs=10
Fold 1/5
Class weights for fold 1: tensor([ 0.5473,  5.0240,  6.9462,  7.7194, 10.4045], device='cuda:0')


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Model: xlm-roberta-base
Number of parameters: 278241797
Tokenizer vocab size: 250002


Evaluating: 100%|██████████| 157/157 [00:17<00:00,  9.20it/s]


Epoch 1: Train Loss=0.8413, Val F1=0.6346


...

Evaluating: 100%|██████████| 79/79 [00:16<00:00,  4.83it/s]


Epoch 4: Train Loss=0.3670, Val F1=0.7669


Evaluating: 100%|██████████| 79/79 [00:16<00:00,  4.81it/s]


Epoch 5: Train Loss=0.3057, Val F1=0.7867


Evaluating:  11%|█▏        | 9/79 [00:01<00:14,  4.82it/s]